In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('laptop_price.csv',encoding='latin-1')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df=df.drop('laptop_ID',axis=1)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df['Ram']=df['Ram'].str.replace('GB','').astype(int)
df['Weight']=df['Weight'].str.replace('kg','').astype(float)

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
cat_col=['Company','TypeName','ScreenResolution','Cpu','Memory','Gpu','OpSys']

In [ ]:
df=df.drop('Product',axis=1)

In [ ]:
df_encoded=pd.get_dummies(df,columns=cat_col,drop_first=True).astype(int)

In [ ]:
df_encoded.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler=StandardScaler()

In [ ]:
num_col = ['Inches', 'Ram', 'Weight']

In [ ]:
df_encoded[num_col]=scaler.fit_transform(df_encoded[num_col])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x=df_encoded.drop('Price_euros',axis=1)
y=df_encoded['Price_euros']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
import tensorflow as tf

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model=Sequential()

In [ ]:
input_dim=x_train.shape[1]

In [ ]:
x_train.shape

In [ ]:
input_dim=x_train.shape[1]

In [ ]:
model.add(Dense(units=128,activation='relu',input_shape=(input_dim,)))
model.add(Dropout(0.2))

model.add(Dense(units=128,activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(units=64,activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(units=32,activation='relu'))
model.add(Dropout(0.2))




In [ ]:
model.add(Dense(units=1,activation='linear'))

In [ ]:
early=EarlyStopping(monitor='val_loss',verbose=1,patience=5)

In [ ]:
model.compile(optimizer='adam',loss='mean_squared_error',metrics=['mean_absolute_error'])

In [ ]:
history=model.fit(x_train,y_train,epochs=100,validation_split=0.2,callbacks
=[early])

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
y_prd_trin=model.predict(x_train)

In [ ]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
import pickle
import tensorflow as tf

# Create a dictionary to store all necessary components
model_components = {
    'model': model,
    'scaler': scaler,
    'cat_col': cat_col,
    'num_col': num_col,
    'feature_columns': x.columns.tolist() # Save the order of features
}

# Save the dictionary to a .pkl file
with open('laptop_price_prediction_model.pkl', 'wb') as file:
    pickle.dump(model_components, file)

print("Model and related components saved to 'laptop_price_prediction_model.pkl'")

You can load the model and components later using:

```python
import pickle
import tensorflow as tf

with open('laptop_price_prediction_model.pkl', 'rb') as file:
    loaded_components = pickle.load(file)

loaded_model = loaded_components['model']
loaded_scaler = loaded_components['scaler']
loaded_cat_col = loaded_components['cat_col']
loaded_num_col = loaded_components['num_col']
loaded_feature_columns = loaded_components['feature_columns']

print("Model and components loaded successfully!")
```

When loading a Keras model saved with `pickle`, TensorFlow typically reconstructs it. If you encounter issues with `tf.keras.models.load_model` in future versions or different environments, you might need to save and load the Keras model separately using `model.save()` and `tf.keras.models.load_model()` and then include the path to the saved model in your pickled dictionary.

In [ ]:
print(mean_squared_error(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))
print(r2_score(y_test,y_pred))